# GPT-OSS detail representation alignment

This notebook asks whether semantically retained details are represented similarly in an original story and its actual-age summary, whether native record context changes that alignment relative to independent detail encoding, and whether retained and omitted original details differ in contextualization.

Primary scope: `condition == 'actual'` summaries, final-layer last-token GPT-OSS representations, and MPNet matching defined independently of GPT-OSS. A final complete-triplet sensitivity analysis compares `younger`, `actual`, and `older` summaries within the same story.

## What was extracted?

For a detail such as *fox trapped*, the contextual extractor tokenized the complete, untruncated record, located the detail by character offsets, mapped the span to token indices, and saved the hidden state at the detail's final token at layer $\ell$:

$$z^{ctx}_{i,\ell}=h_{\ell,t_i^{end}}(x_{1:t_i^{end}}).$$

Thus $h_\ell(\text{fox trapped}\mid\text{previous story})$ means the representation at the last token of *fox trapped*, conditioned on the preceding record and the detail itself. GPT-OSS is causal, so it does not contain later story tokens. The independently extracted control encodes the detail text alone and saves its final EOS state, $z^{iso}_{i,\ell}=h_{\ell,EOS}(d_i)$. These positions are analogous readouts but not literally the same token position.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get('STORY_FEATURES_ROOT', Path.cwd())).resolve()
if not (ROOT / 'mechanistic_interpretability').exists():
    ROOT = ROOT.parent

CONTEXT_DIR = Path(os.environ.get(
    'GPT_OSS_CONTEXT_DIR',
    ROOT / 'mechanistic_interpretability/outputs/02_representation_extraction/gpt_oss_20b_global_current',
))
ISOLATED_DIR = Path(os.environ.get(
    'GPT_OSS_ISOLATED_DIR',
    ROOT / 'mechanistic_interpretability/outputs/02_representation_extraction/gpt_oss_20b_detail_controls',
))
MPNET_DIR = Path(os.environ.get(
    'MPNET_ALIGNMENT_DIR',
    ROOT / 'mechanistic_interpretability/outputs/13_structured_detail_alignment',
))
AGE_CENTER = 8.0
CONDITION_ORDER = ['younger', 'actual', 'older']
print('Contextual vectors:', CONTEXT_DIR)
print('Isolated vectors:', ISOLATED_DIR)
print('MPNet matches:', MPNET_DIR)

## RQ1 — Which observations have contextual and isolated representations?

The MPNet table supplies, for every original detail, its most similar summary detail (`best_summary_idx`), similarity (`best_cosine`), and thresholded retention label (`retained_cosine`, default MPNet threshold 0.35). GPT-OSS is not used to choose the pair.

In [ ]:
def normalized_rows(values):
    values = np.asarray(values, dtype=np.float32)
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    return values / np.maximum(norms, 1e-12)

def parse_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    parsed = series.astype(str).str.lower().map({'true': True, 'false': False})
    if parsed.isna().any():
        raise ValueError('Could not parse a Boolean retention column.')
    return parsed.astype(bool)

def canonical_summary_id(series):
    numeric = pd.to_numeric(series, errors='coerce')
    return numeric.astype('Int64').astype(str)

def input_csv(name):
    merged = MPNET_DIR / name
    if merged.exists():
        return [merged]
    shards = sorted(MPNET_DIR.glob(f'shard_*/{name}'))
    if not shards:
        raise FileNotFoundError(f'No {name} under {MPNET_DIR}')
    return shards

mpnet = pd.concat(
    [pd.read_csv(path, low_memory=False) for path in input_csv('original_detail_retention.csv')],
    ignore_index=True,
)
required = {'story_key', 'condition', 'summary_id', 'source_age', 'orig_idx',
            'orig_text', 'best_summary_idx', 'best_cosine', 'retained_cosine'}
missing = sorted(required - set(mpnet.columns))
if missing:
    raise ValueError(f'MPNet table lacks columns: {missing}')
mpnet = mpnet.loc[mpnet['condition'].astype(str).isin(CONDITION_ORDER)].copy()
mpnet['summary_id_key'] = canonical_summary_id(mpnet['summary_id'])
mpnet['retained'] = parse_bool(mpnet['retained_cosine'])
mpnet['source_age'] = pd.to_numeric(mpnet['source_age'], errors='coerce')
mpnet['mpnet_similarity'] = pd.to_numeric(mpnet['best_cosine'], errors='coerce')
mpnet['original_detail_index'] = pd.to_numeric(mpnet['orig_idx'], errors='raise').astype(int)
mpnet['summary_detail_index'] = pd.to_numeric(mpnet['best_summary_idx'], errors='raise').astype(int)
print(f'Three-condition MPNet rows: {len(mpnet):,}')
print(f'Stories: {mpnet.story_key.nunique():,}')
print(f'Retention rate: {mpnet.retained.mean():.3f}')

In [ ]:
def load_contextual(directory):
    metadata_blocks, vector_blocks = [], []
    for metadata_path in sorted(directory.glob('contextual_detail_metadata_*.csv')):
        suffix = metadata_path.name.removeprefix('contextual_detail_metadata_').removesuffix('.csv')
        feature_path = directory / f'contextual_detail_features_{suffix}.npz'
        if not feature_path.exists():
            continue
        metadata = pd.read_csv(metadata_path, low_memory=False)
        keep = metadata['record_type'].astype(str).eq('original') | (
            metadata['record_type'].astype(str).eq('summary')
            & metadata['condition'].astype(str).isin(CONDITION_ORDER)
        )
        positions = np.flatnonzero(keep.to_numpy())
        if not len(positions):
            continue
        with np.load(feature_path, allow_pickle=False) as payload:
            layers = payload['layers']
            layer_position = int(np.argmax(layers))
            vectors = normalized_rows(payload['last_token_vectors'][positions, layer_position])
        selected = metadata.iloc[positions].copy().reset_index(drop=True)
        selected['vector_index'] = np.arange(len(vectors)) + sum(len(x) for x in vector_blocks)
        selected['layer'] = int(layers[layer_position])
        metadata_blocks.append(selected)
        vector_blocks.append(vectors)
    if not metadata_blocks:
        raise FileNotFoundError(f'No contextual GPT-OSS shards in {directory}')
    return pd.concat(metadata_blocks, ignore_index=True), np.vstack(vector_blocks)

def load_isolated(directory):
    metadata_blocks, vector_blocks = [], []
    for metadata_path in sorted(directory.glob('isolated_detail_metadata_*.csv')):
        suffix = metadata_path.name.removeprefix('isolated_detail_metadata_').removesuffix('.csv')
        feature_path = directory / f'isolated_detail_features_{suffix}.npz'
        if not feature_path.exists():
            continue
        metadata = pd.read_csv(metadata_path, low_memory=False)
        keep = metadata['record_type'].astype(str).eq('original') | (
            metadata['record_type'].astype(str).eq('summary')
            & metadata['condition'].astype(str).isin(CONDITION_ORDER)
        )
        positions = np.flatnonzero(keep.to_numpy())
        if not len(positions):
            continue
        with np.load(feature_path, allow_pickle=False) as payload:
            layers = payload['layers']
            layer_position = int(np.argmax(layers))
            vectors = normalized_rows(payload['eos_vectors'][positions, layer_position])
        selected = metadata.iloc[positions].copy().reset_index(drop=True)
        selected['isolated_vector_index'] = np.arange(len(vectors)) + sum(len(x) for x in vector_blocks)
        selected['isolated_layer'] = int(layers[layer_position])
        metadata_blocks.append(selected)
        vector_blocks.append(vectors)
    if not metadata_blocks:
        raise FileNotFoundError(f'No isolated GPT-OSS shards in {directory}')
    return pd.concat(metadata_blocks, ignore_index=True), np.vstack(vector_blocks)

context_meta, context_vectors = load_contextual(CONTEXT_DIR)
isolated_meta, isolated_vectors = load_isolated(ISOLATED_DIR)
print(f'Contextual detail vectors loaded: {len(context_meta):,}')
print(f'Isolated detail vectors loaded: {len(isolated_meta):,}')
print('Contextual layers:', sorted(context_meta.layer.unique()))
print('Isolated layers:', sorted(isolated_meta.isolated_layer.unique()))

In [ ]:
context_meta['summary_id_key'] = canonical_summary_id(context_meta['summary_id'])
original_records = (context_meta.loc[context_meta.record_type.eq('original'), ['story_key', 'record_id']]
                    .drop_duplicates())
if original_records.story_key.duplicated().any():
    raise ValueError('More than one contextual original record exists for a story.')
summary_records = (context_meta.loc[context_meta.record_type.eq('summary'),
                    ['story_key', 'condition', 'summary_id_key', 'record_id']].drop_duplicates())
summary_key = ['story_key', 'condition', 'summary_id_key']
if summary_records.duplicated(summary_key).any():
    raise ValueError('Actual summaries are not unique by story_key and summary_id; add variant identity.')

pairs = mpnet.merge(original_records.rename(columns={'record_id': 'original_record_id'}),
                     on='story_key', how='inner', validate='many_to_one')
pairs = pairs.merge(summary_records.rename(columns={'record_id': 'summary_record_id'}),
                    on=summary_key, how='inner', validate='many_to_one')

ctx_key = context_meta[['record_id', 'detail_index', 'detail_text', 'vector_index']].copy()
pairs = pairs.merge(
    ctx_key.rename(columns={'record_id': 'original_record_id', 'detail_index': 'original_detail_index',
                            'detail_text': 'gpt_original_text', 'vector_index': 'original_ctx_index'}),
    on=['original_record_id', 'original_detail_index'], how='inner', validate='many_to_one')
pairs = pairs.merge(
    ctx_key.rename(columns={'record_id': 'summary_record_id', 'detail_index': 'summary_detail_index',
                            'detail_text': 'gpt_summary_text', 'vector_index': 'summary_ctx_index'}),
    on=['summary_record_id', 'summary_detail_index'], how='inner', validate='many_to_one')

iso_key = isolated_meta[['record_id', 'detail_index', 'isolated_vector_index']].copy()
pairs = pairs.merge(
    iso_key.rename(columns={'record_id': 'original_record_id', 'detail_index': 'original_detail_index',
                            'isolated_vector_index': 'original_iso_index'}),
    on=['original_record_id', 'original_detail_index'], how='left', validate='many_to_one')
pairs = pairs.merge(
    iso_key.rename(columns={'record_id': 'summary_record_id', 'detail_index': 'summary_detail_index',
                            'isolated_vector_index': 'summary_iso_index'}),
    on=['summary_record_id', 'summary_detail_index'], how='left', validate='many_to_one')

octx = pairs.original_ctx_index.to_numpy(int)
sctx = pairs.summary_ctx_index.to_numpy(int)
pairs['contextual_alignment'] = np.einsum('ij,ij->i', context_vectors[octx], context_vectors[sctx])
pairs['original_context_isolated_cosine'] = np.nan
pairs['isolated_alignment'] = np.nan
has_original_iso = pairs.original_iso_index.notna().to_numpy()
oi = pairs.loc[has_original_iso, 'original_iso_index'].to_numpy(int)
pairs.loc[has_original_iso, 'original_context_isolated_cosine'] = np.einsum(
    'ij,ij->i', context_vectors[octx[has_original_iso]], isolated_vectors[oi])
has_pair_iso = (pairs.original_iso_index.notna() & pairs.summary_iso_index.notna()).to_numpy()
oi = pairs.loc[has_pair_iso, 'original_iso_index'].to_numpy(int)
si = pairs.loc[has_pair_iso, 'summary_iso_index'].to_numpy(int)
pairs.loc[has_pair_iso, 'isolated_alignment'] = np.einsum(
    'ij,ij->i', isolated_vectors[oi], isolated_vectors[si])
pairs['alignment_change'] = pairs.contextual_alignment - pairs.isolated_alignment
pairs['source_age_centered'] = pairs.source_age - AGE_CENTER

display(pd.DataFrame({
    'measure': ['MPNet actual details', 'joined contextual pairs', 'pairs with both isolated vectors',
                'retained contextual pairs', 'retained pairs with isolated vectors'],
    'n': [len(mpnet), len(pairs), int(has_pair_iso.sum()), int(pairs.retained.sum()),
          int((pairs.retained & pairs.isolated_alignment.notna()).sum())],
}))

## RQ2 — In actual-age summaries, how aligned are retained original and summary representations?

`contextual_alignment` is the cosine between the native original-position and summary-position vectors. `isolated_alignment` is the cosine between the two independently encoded detail texts. `alignment_change = contextual_alignment - isolated_alignment`; a negative value means the two record contexts pull the pair apart relative to standalone encoding.

In [ ]:
actual_pairs = pairs.loc[pairs.condition.astype(str).eq('actual')].copy()
retained_pairs = actual_pairs.loc[actual_pairs.retained].copy()
display(retained_pairs[['mpnet_similarity', 'contextual_alignment', 'isolated_alignment',
                        'alignment_change']].describe().T)
retained_pairs.plot.scatter(x='source_age', y='alignment_change', alpha=0.12, figsize=(7, 4))
plt.axhline(0, color='black', linewidth=1)
plt.title('Actual-age summaries: contextual minus isolated alignment')
plt.show()

## RQ3 — In actual-age summaries, do retained and omitted original details differ in contextualization?

Every original detail can be compared with its own independent encoding. This supports a retained-versus-omitted comparison even though omitted details have no defensible summary counterpart. Lower cosine means the original story prefix changed the detail representation more strongly.

In [ ]:
display(actual_pairs.groupby('retained').agg(
    details=('original_detail_index', 'size'),
    stories=('story_key', 'nunique'),
    mean_context_isolated=('original_context_isolated_cosine', 'mean'),
    sd_context_isolated=('original_context_isolated_cosine', 'std'),
))
pairs.boxplot(column='original_context_isolated_cosine', by='retained', showfliers=False)
plt.suptitle('')
plt.title('Original contextualization by MPNet retention')
plt.xlabel('Retained in actual-age summary')
plt.ylabel('cos(contextual original, isolated original)')
plt.show()

## RQ4 — Confirmatory mixed-effects tests

Model 1 tests whether native contextual alignment changes with source age among retained details while controlling for MPNet textual similarity. Model 2 repeats the test for contextual-minus-isolated alignment in the subset with both isolated controls. Model 3 tests the retained-versus-omitted difference in original contextualization and whether that difference changes with source age. All include a random intercept for story. Since only actual-age summaries are included, there is no condition term.

In [ ]:
try:
    import statsmodels.formula.api as smf
except ImportError as error:
    raise ImportError('This cell requires statsmodels in the notebook kernel.') from error

contextual_model_data = retained_pairs.dropna(subset=[
    'contextual_alignment', 'source_age_centered', 'mpnet_similarity', 'story_key'
]).copy()
contextual_alignment_model = smf.mixedlm(
    'contextual_alignment ~ source_age_centered + mpnet_similarity',
    contextual_model_data, groups=contextual_model_data['story_key'],
).fit(reml=False, method='lbfgs')
print('MODEL 1: retained-pair contextual alignment')
print(contextual_alignment_model.summary())

change_model_data = retained_pairs.dropna(subset=[
    'alignment_change', 'source_age_centered', 'mpnet_similarity', 'story_key'
]).copy()
alignment_change_model = smf.mixedlm(
    'alignment_change ~ source_age_centered + mpnet_similarity',
    change_model_data, groups=change_model_data['story_key'],
).fit(reml=False, method='lbfgs')
print('\nMODEL 2: retained-pair contextual minus isolated alignment')
print(alignment_change_model.summary())

context_model_data = actual_pairs.dropna(subset=[
    'original_context_isolated_cosine', 'source_age_centered', 'retained', 'story_key'
]).copy()
context_model_data['retained_label'] = pd.Categorical(
    np.where(context_model_data.retained, 'retained', 'omitted'),
    categories=['omitted', 'retained'],
)
context_model = smf.mixedlm(
    'original_context_isolated_cosine ~ source_age_centered * C(retained_label)',
    context_model_data, groups=context_model_data['story_key'],
).fit(reml=False, method='lbfgs')
print('\nMODEL 3: retained versus omitted original contextualization')
print(context_model.summary())

In [ ]:
age_grid = np.linspace(
    context_model_data.source_age.min(), context_model_data.source_age.max(), 100
)
prediction_base = pd.DataFrame({
    'source_age': age_grid,
    'source_age_centered': age_grid - AGE_CENTER,
    'mpnet_similarity': contextual_model_data.mpnet_similarity.mean(),
})
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(contextual_model_data.source_age, contextual_model_data.contextual_alignment,
                alpha=0.06, s=10)
axes[0].plot(age_grid, contextual_alignment_model.predict(prediction_base), color='black', linewidth=2)
axes[0].set(title='Retained-pair contextual alignment', xlabel='Source age', ylabel='Cosine')
axes[1].scatter(change_model_data.source_age, change_model_data.alignment_change,
                alpha=0.06, s=10)
axes[1].axhline(0, color='grey', linewidth=1)
axes[1].plot(age_grid, alignment_change_model.predict(prediction_base), color='black', linewidth=2)
axes[1].set(title='Contextual minus isolated alignment', xlabel='Source age', ylabel='Cosine difference')
for label, color in [('omitted', 'tab:orange'), ('retained', 'tab:blue')]:
    grid = pd.DataFrame({
        'source_age_centered': age_grid - AGE_CENTER,
        'retained_label': pd.Categorical([label] * len(age_grid),
                                         categories=['omitted', 'retained']),
    })
    axes[2].plot(age_grid, context_model.predict(grid), label=label, color=color, linewidth=2)
axes[2].set(title='Original contextualization', xlabel='Source age',
            ylabel='cos(contextual, isolated)')
axes[2].legend(title='MPNet status')
plt.tight_layout()
plt.show()

## RQ5 — Within stories having all three summaries, how large are the representation effects?

For each summary, detail-level outcomes are first aggregated to one value. The analysis then retains only stories with `younger`, `actual`, and `older` values. Each condition contrast reports its raw paired difference, 95% CI, paired *t* test, and paired Cohen's $d_z=\bar{D}/s_D$. Holm correction is applied separately within each representation outcome.

In [ ]:
from scipy.stats import t as student_t, ttest_1samp

summary_metric_rows = []
summary_keys = ['summary_record_id', 'story_key', 'condition', 'summary_id', 'source_age']
for identity, frame in pairs.groupby(summary_keys, sort=False, dropna=False):
    retained = frame.retained.to_numpy(bool)
    contextual_shift = 1.0 - frame.original_context_isolated_cosine
    row = dict(zip(summary_keys, identity))
    row['retained_contextual_alignment'] = frame.loc[retained, 'contextual_alignment'].mean()
    row['retained_alignment_change'] = frame.loc[retained, 'alignment_change'].mean()
    retained_shift = contextual_shift[retained].dropna()
    omitted_shift = contextual_shift[~retained].dropna()
    row['retained_minus_omitted_contextualization'] = (
        retained_shift.mean() - omitted_shift.mean()
        if len(retained_shift) and len(omitted_shift) else np.nan
    )
    summary_metric_rows.append(row)
summary_metrics = pd.DataFrame(summary_metric_rows)

representation_outcomes = [
    'retained_contextual_alignment',
    'retained_alignment_change',
    'retained_minus_omitted_contextualization',
]
paired_conditions = [('younger', 'actual'), ('older', 'actual'), ('older', 'younger')]
effect_rows, triplet_tables = [], {}
for outcome in representation_outcomes:
    story_condition = summary_metrics.groupby(
        ['story_key', 'condition'], observed=True, as_index=False
    ).agg(value=(outcome, 'mean'))
    triplet = story_condition.pivot(
        index='story_key', columns='condition', values='value'
    ).reindex(columns=CONDITION_ORDER).dropna()
    triplet_tables[outcome] = triplet
    for left, right in paired_conditions:
        difference = (triplet[left] - triplet[right]).to_numpy(float)
        n = len(difference)
        mean_difference = float(np.mean(difference)) if n else np.nan
        sd_difference = float(np.std(difference, ddof=1)) if n > 1 else np.nan
        se_difference = sd_difference / np.sqrt(n) if n > 1 else np.nan
        critical = float(student_t.ppf(0.975, n - 1)) if n > 1 else np.nan
        test = ttest_1samp(difference, 0.0) if n > 1 else None
        effect_rows.append({
            'outcome': outcome, 'contrast': f'{left} - {right}', 'stories': n,
            'mean_paired_difference': mean_difference,
            'ci95_low': mean_difference - critical * se_difference,
            'ci95_high': mean_difference + critical * se_difference,
            'cohens_dz': mean_difference / sd_difference if sd_difference > 0 else np.nan,
            't': float(test.statistic) if test is not None else np.nan,
            'p_value': float(test.pvalue) if test is not None else np.nan,
        })
representation_effects = pd.DataFrame(effect_rows)
representation_effects['p_holm'] = np.nan
for outcome, indices in representation_effects.groupby('outcome').groups.items():
    indices = np.asarray(list(indices), dtype=int)
    p_values = representation_effects.loc[indices, 'p_value'].to_numpy()
    order = np.argsort(p_values)
    adjusted_sorted = np.maximum.accumulate((len(order) - np.arange(len(order))) * p_values[order])
    adjusted = np.empty(len(order))
    adjusted[order] = np.minimum(adjusted_sorted, 1.0)
    representation_effects.loc[indices, 'p_holm'] = adjusted
display(representation_effects)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
x = np.arange(len(CONDITION_ORDER))
titles = ['Retained contextual alignment', 'Contextual − isolated alignment',
          'Retained − omitted contextualization']
for ax, outcome, title in zip(axes, representation_outcomes, titles):
    triplet = triplet_tables[outcome]
    for _, row in triplet.iterrows():
        ax.plot(x, row.to_numpy(float), color='grey', alpha=0.10, linewidth=0.7)
    ax.plot(x, triplet.mean().to_numpy(float), color='black', marker='o', linewidth=2.5)
    ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
    ax.set_xticks(x, CONDITION_ORDER)
    ax.set_title(f'{title}\n(n={len(triplet)} complete stories)')
    ax.set_xlabel('Summary condition')
plt.tight_layout()
plt.show()

## Interpretation boundaries

- MPNet defines textual matching and retention; GPT-OSS supplies the outcomes, avoiding selection on the outcome.
- `alignment_change` is a cosine difference, not a percentage of information lost.
- The contextual vector is prefix-conditioned at the detail position, not bidirectionally conditioned on the complete record.
- The isolated control uses an EOS readout, whereas the native contextual vector uses the detail's final token. Interpret their difference as a controlled representation comparison, not a perfectly position-matched causal effect.